<a href="https://colab.research.google.com/github/Steve-Falkovsky/Hypencoder-Entity-Linking/blob/Professional-Structure/notebooks/fine_tune_Hypencoder_on_BC5CDR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Environment lock (run once per fresh runtime) ---
# because libraries like transformers can change and make your code non-runnable :)
# Torch updates also broke the code but installing it in colab to an older version
# takes 2 minutes every time so I just solved the one bug

!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/env_setup.py
import env_setup
env_setup.install_env()

In [ ]:
import os

REPO_NAME = "Hypencoder-Entity-Linking"
GIT_URL = f"https://github.com/Steve-Falkovsky/{REPO_NAME}.git"
MAIN_BRANCH_NAME = "main"
OLD_PYTORCH_BRANCH_NAME = "oldtorch"

!git clone -b {MAIN_BRANCH_NAME} --single-branch {GIT_URL}

# Move into the downloaded repo (The Root)
os.chdir(REPO_NAME)

%pip install -q -e "./hypencoder-paper"
%pip install fire jsonlines omegaconf # dependencies for the training

os.chdir("./hypencoder-paper")

print(f"📍 Working Directory is now: {os.getcwd()}")
print("✅ Environment Ready!")

In [ ]:
# loading the data
from datasets import load_dataset

# change the dataset based on which one you want to use
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")

train_data = dataset['train']
val_data = dataset['validation']

# We need to put this data in a format that the hypencoder code needs:

In [ ]:
"""
create JSONL files for contrastive loss training from BC5CDR and MeSH2015 datasets to be used with Hypencoder

Desired format for each line in the JSONL file:
{
  "query": {
    "id": query ID,
    "content": <mention_name> [SEP] <mention_text_window>,
  },
  "items": [
    {
      "id": passage ID,
      "content": <entity_name> [SEP] <definition[:128]>,
      "score": Optional teacher score,
      "type": Sometimes used to specify type of item,
    },
  ]
}

This is contrastive Loss without Hard Negatives: just include only a single positive i.e. relevant item to the query in the "items" for that query.
The negatives will then be the other queries positives in the batch.
"""

In [ ]:
# Prepare Data

# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window


def format_to_hypencoder_json(dataset):
    """Formats the dataset into a list of dictionaries ready for JSONL."""
    formatted_data = []

    # Using enumerate to create arbitrary unique IDs
    for idx, (m, mt, e, d) in enumerate(zip(
        dataset["mention"],
        dataset["mention_text"],
        dataset["entity"],
        dataset["definition"],
    )):
        # Pre-processing
        mt_window = get_mt_window(m, mt)
        d_short = d[:128]

        # Construct the Hypencoder structure
        entry = {
            "query": {
                "id": f"q_{idx}",
                "content": f"{m} [SEP] {mt_window}"
            },
            "items": [
                {
                    "id": f"p_{idx}",
                    "content": f"{e} [SEP] {d_short}",
                    "score": None,
                    "type": None,
                }
            ]
        }
        formatted_data.append(entry)

    return formatted_data


In [ ]:
train_json_list = format_to_hypencoder_json(train_data)
val_json_list = format_to_hypencoder_json(val_data)

In [ ]:
import json

def save_as_jsonl(data_list, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for entry in data_list:
            f.write(json.dumps(entry) + '\n')

In [ ]:
import os

os.makedirs('data', exist_ok=True)
save_as_jsonl(train_json_list, "data/train.jsonl")
save_as_jsonl(val_json_list, "data/val.jsonl")

In [ ]:
# tokenizing the data before training
# change the tokenizer based on your model

# bert = "google-bert/bert-base-uncased"
# sapbert = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
# (for some reason "fire" needs the literal string and will not accept it as a variable)


# training
!python hypencoder_cb/utils/tokenizer_utils.py \
--standard_format_jsonl='data/train.jsonl' \
--output_file='data/train_tokenized.jsonl' \
--tokenizer="cambridgeltl/SapBERT-from-PubMedBERT-fulltext" \
--add_special_tokens=True \
--query_max_length=32 \
--item_max_length=512

# validation
!python hypencoder_cb/utils/tokenizer_utils.py \
--standard_format_jsonl='data/val.jsonl' \
--output_file='data/val_tokenized.jsonl' \
--tokenizer="cambridgeltl/SapBERT-from-PubMedBERT-fulltext" \
--add_special_tokens=True \
--query_max_length=32 \
--item_max_length=512



---



# Training the hypencoder
The 2nd argument should be the path to the model configuration you want to train

In [ ]:
!python hypencoder_cb/train/train.py hypencoder_cb/train/configs/hypencoder.2_layer_SapBERT.yaml
# hard negatives
# !python hypencoder_cb/train/train.py hypencoder_cb/train/configs/hypencoder.2_layer_SapBERT_hardneg.yaml

In [ ]:
# push the model to HuggingFace
from huggingface_hub import upload_folder

# Path where the model files are saved in Colab
# check and change this based on which checkpoint you got
local_folder_path = "model/hypencoder.2_layer_SapBERT/checkpoint-860"

In [ ]:
repo_id = "Stevenf232/SapBERT_freeze_hypencoder_context"

# You may need to create the repository first if it doesn't exist
from huggingface_hub import create_repo
create_repo(repo_id, exist_ok=True)

upload_folder(
    folder_path=local_folder_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload trained model from Colab"
)